In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftConfig
from huggingface_hub import login

from langgraph.graph import StateGraph, START, END
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import Literal, List ,Optional, TypedDict, Any
from pathlib import Path
from asgiref.sync import sync_to_async

import os, sys, threading

In [ ]:
model=ChatOllama(model='')

In [ ]:
class InputModel(BaseModel):
    flaws:Optional[List[str]]=Field(default=None, description="Specific problems or issues mentioned in the reviews. Only include what customers actually complained about.")
    strengths:Optional[List[str]]=Field(default=None, description="Good features or benefits mentioned in the reviews. Only include what customers actually praised.")
    overall_rating:Optional[float]=Field(default=None, description="Rating from 0 to 5. Use 0=very bad, 1=bad, 2=poor, 3=average, 4=good, 5=excellent. Choose one number based on overall sentiment.")

In [ ]:
class ResponseModel(BaseModel):
    places_to_fix:Optional[List[str]]=Field(default=None, description="List specific areas or components that need improvement. Match these to the flaws mentioned.")
    Recommendations:Optional[List[str]]=Field(default=None, description="Suggest specific, practical actions to fix each problem. Be clear and direct.")

In [ ]:
class StateSchema(TypedDict):
    raw_reviews:Optional[str]
    doc_text:Optional[str]=""
    product_name:Optional[str]
    flaws:Optional[List[str]]
    strengths:Optional[List[str]]
    weaknesses:Optional[List[str]]
    product_rating:Optional[float]
    places_to_fix:Optional[List[str]]
    recommendations:Optional[List[str]]

In [ ]:
vector_stores=Chroma(collection_name="model_collections", embedding_function=OllamaEmbeddings(model=""), persist_directory='vector_stores')
model=ChatOllama(model="")

In [ ]:
data_folder=r'D:\SAKETH\Auto_insighter\project\project2\apps\api\app\myApp\Cache'
doc_text=''
files=os.listdir(data_folder)
if(len(files)==1):
    file=files[0]
    print(file)
    loader=None
    if(file.endswith(".pdf")):
        loader=PyPDFLoader(f'{data_folder}/{file}')
    elif(file.endswith(".docx")):
        loader=Docx2txtLoader(f'{data_folder}/{file}')
    elif(file.endswith(".txt")):
        loader=TextLoader(f'{data_folder}/{file}')
    if loader is not None:
        docs=loader.load()
        doc_text="\n".join(doc.page_content for doc in docs)

In [ ]:
def validate(schema:StateSchema):
    llm=model.with_structured_output(InputModel)
    reviews_text = schema['raw_reviews'][:14000]
    
    response=llm.invoke(
        f"""TASK: Extract product information from customer reviews.

PRODUCT: {schema['product_name']}

REVIEWS:
{reviews_text}

INSTRUCTIONS:
1. Read the reviews carefully.
2. Extract FLAWS - what customers complained about (actual problems mentioned)
3. Extract STRENGTHS - what customers liked (actual benefits mentioned)
4. Extract RATING - the overall sentiment rating 0-5 (0=very bad, 5=excellent)

Focus ONLY on what is actually mentioned in the reviews. Do not make up information.
If a field is not mentioned, leave it empty."""
    )
    print(f"Validate response: {response}")
    schema['flaws']=response.flaws if response.flaws else ["no flaws seen"]
    schema['strengths']=response.strengths if response.strengths else ["the prduct has no strengths"]
    schema['product_rating']=response.overall_rating if response.overall_rating else 0.0

    return schema

def fix(schema:StateSchema):
    llm=model.with_structured_output(ResponseModel)
    prompt=f"""TASK: Generate solutions to fix product problems.

PRODUCT: {schema.get('product_name')}

PROBLEMS TO FIX, BY SEEING THIS GENERATE THE PLACES TO FIX THIS IS THE CORE PART FOR PLACES TO FIX:
{schema.get('flaws')}

POSITIVE ASPECTS TO MAINTAIN:
{schema.get('strengths')}

CURRENT RATING: {schema.get('product_rating')}/5

INSTRUCTIONS:
1. For each problem listed, suggest a specific fix or improvement
2. Write clear, actionable recommendations
3. Focus on realistic solutions based on the problems mentioned

Generate fixes that directly address the listed problems."""
    
    response=llm.invoke(prompt)
    print(f"Fix response: {response}")
    schema['places_to_fix']=response.places_to_fix if response.places_to_fix else ["no place to fix, the product is just fine"]
    schema['recommendations']=response.Recommendations if response.Recommendations else ["not much recommendations are neccessary as the product is already heavily optimized"]
    return schema

In [ ]:
graph=StateGraph(StateSchema)
graph.add_node("validate", validate)
graph.add_node("fix", fix)

graph.add_edge(START, "validate")
graph.add_edge("validate", "fix")
graph.add_edge("fix", END)

workflow=graph.compile()

In [ ]:
if __name__ == "__main__":
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

    from apps.api.app.myApp.app_views.rough import get_rows
    from apps.scrapper.review_scraper import get_product_reviews

    table_rows = await sync_to_async(get_rows)()
    product_name=table_rows[-1].human_query
    eg_product="PS5 gaming console"
    scraped_content=await sync_to_async(get_product_reviews)(product_name)
    
    print(f"Product name: {product_name}")
    print(f"Scraped content length: {len(scraped_content)}")
    print(f"First 500 chars of scraped content: {scraped_content[:500]}")
    
    model_response=workflow.invoke({'raw_reviews':scraped_content, 'product_name':product_name, 'doc_text':""})
    print(model_response)

In [ ]:
print(model_response['flaws'])
print(model_response['places_to_fix'])
print(model_response['product_rating'])
print(model_response['recommendations'])
print(model_response['strengths'])

In [ ]:
model_text = f"""
Product_name: {model_response["product_name"]}

Product rating: {model_response["product_rating"]}

Pros:

{chr(10).join(pro for pro in model_response["strengths"])}

Cons:

{chr(10).join(con for con in model_response['flaws'])}

Places that you need to look after:

{chr(10).join(place for place in model_response['places_to_fix'])}

Further Recommendations:

{chr(10).join(recomm for recomm in model_response['recommendations'])}
"""

model_text

In [ ]:
print(model_text)